# Silver Layer
--------------
- Clean the data
- Make sure data is ready to add BI logic


#Import Libraries
------------------

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
import requests
from utill_funcs import get_winner
import plotly.express as px

#Parameters
--------------

In [0]:
catalog = 'wc_2026_predions'
bronze_schema = 'bronze'
silver_schema = 'silver'
bronze_table_name = 'source'
silver_table_name = 'cleaned_data'

#Read in Bronze Table
---------------------
- Read in the source table saved in unity catalog (wc_2026_predictions)

In [0]:
df_bronze = spark.read.table(f'{catalog}.{bronze_schema}.{bronze_table_name}')
display(df_bronze)

#check for duplications & nulls
--------------------------

In [0]:
### Check num nulls per columns
display(df_bronze.select([sum(col(c).isNull().cast("int")).alias(c) for c in df_bronze.columns]))

In [0]:
# check the nulls only
df_bronze.filter(col("home_score").isNull() | col("away_score").isNull()).createOrReplaceTempView("null_scores_view")
display(spark.table("null_scores_view"))

# Make date field a date type
-----------------------------------

In [0]:
df_bronze = df_bronze.withColumn('date', to_date(col('date'), 'yyyy-mm-dd'))

In [0]:
display(df_bronze)

#Write the silver table
---------------------------

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{silver_schema}")

In [0]:
df_bronze.write.format("delta").mode("overwrite").saveAsTable(f'{catalog}.{silver_schema}.{silver_table_name}')